In [1]:
# key realization: we don't have to spend all of our money!

import numpy as np

def rs_given_speed(speed_stat: float, v: int):
    best_r_s = (0, 0)
    best_profit = -1
    for r in range(100 - v):
        for s in range(100 - v - s):
            profit = 200000 * np.log(1 + r) / np.log(101) * 0.07 * s * speed_stat
            if profit > best_profit:
                best_r_s = (r, s)
                best_profit = profit
    return best_r_s

In [2]:
# generate prior of users
# 60% - 70%: balanced
#  2% -  4%: research (these guys are stupid)
#  8% - 12%: scale (same here)
# 15% - 20%: speedy as f
#  2% -  4%: random (they're not very smart lol)

# assumptions:
# all people will use between 90% - 100% of their money, vast majority using all of it
# base spreads across different types of people

def generate_trader(idx: int):
    # allocate %age of investment
    percentages = list(range(90, 101, 1))
    percentage_probs = [0.9 * 0.1 ** (10 - i) for i in range(11)]
    sum_pp = sum(percentage_probs)
    for i in range(len(percentage_probs)):
        percentage_probs[i] /= sum_pp
    total_alloc = np.random.choice(percentages, p=percentage_probs)

    base_spreads = [np.array([20, 50, 30]), 
                    np.array([35, 40, 25]), 
                    np.array([15, 65, 20]), 
                    np.array([15, 35, 50]), 
                    np.array([33, 33, 34])]
    base_spread = base_spreads[idx]
    for _ in range(100 - total_alloc):
        cut_idx = np.random.choice(3)
        base_spread[cut_idx] -= 1

    deviation_spread = np.random.choice(list(range(-7, 8)), size=3)
    while deviation_spread.sum() != 0:
        deviation_spread = np.random.choice(list(range(-7, 8)), size=3)

    final_spread = base_spread + deviation_spread
    return final_spread

In [3]:
# simulate the final outcome

def get_trader_ranks(traders: list[np.ndarray]):
    scale_dict = {}
    for trader in traders:
        scale = int(trader[2])
        if scale not in scale_dict:
            scale_dict[scale] = 0
        scale_dict[scale] += 1
    
    scale_list = []
    for key, val in scale_dict.items():
        scale_list.append([key, val])
    scale_list.sort(key = lambda x: -x[0])

    cur_rank, max_rank = 1, 1
    ranks = {}
    for l in scale_list:
        ranks[l[0]] = cur_rank
        max_rank = cur_rank
        cur_rank += l[1]
    return ranks, max_rank

def calculate_trader_scores(traders: list[np.ndarray]):
    ranks, max_rank = get_trader_ranks(traders)
    slope = -0.8 / (max_rank - 1)

    # 1 --> 0.9; mr --> 0.1
    # slope = (0.1 - 0.9) / (mr - 1)
    # y - 0.9 = slope (x - 1)
    # y = (0.9 - slope) + slope * x

    result = []
    for trader in traders:
        research = 200000 * np.log(1 + trader[0]) / np.log(1 + 100)
        scale = 7 * trader[1] / 100
        speed = (0.9 - slope) + slope * ranks[trader[2]]
        # print(research, "\t", scale, "\t", speed)
        if not (0.1 <= speed <= 0.9):
            speed = max(min(speed, 0.9), 0.1) # PYTHON IS KHOOOOOOSE
        assert(research >= 0)
        assert(0.1 <= speed <= 0.9)
        result.append(research * scale * speed - 50000 * trader.sum() / 100)
    return result

In [4]:
# find optimal traders for different populations, see what they look like

from copy import deepcopy

def generate_traders(trader_cnt: int):
    traders = []
    for _ in range(trader_cnt):
        people_dist = [np.random.uniform(0.60, 0.70),
                       np.random.uniform(0.02, 0.04),
                       np.random.uniform(0.08, 0.12),
                       np.random.uniform(0.15, 0.20),
                       np.random.uniform(0.02, 0.04)]
        sum_pd = sum(people_dist)
        for i in range(5):
            people_dist[i] /= sum_pd
        trader_idx = np.random.choice(5, p=people_dist)
        traders.append(generate_trader(trader_idx))
    return traders

def find_optimal_trader(traders: list[np.ndarray]):
    best_profit = -1e9
    best_trader = np.ndarray([0, 0, 0])

    for r in range(100):
        for s in range(100 - r):
            for v in range(100 - r - s):
                cur_trader = np.array([r, s, v])
                traders_copy = deepcopy(traders)
                traders_copy.append(cur_trader)
                cur_trader_profit = calculate_trader_scores(traders_copy)[-1]

                if cur_trader_profit > best_profit:
                    best_profit = cur_trader_profit
                    best_trader = cur_trader
    return best_trader, best_profit

In [ ]:
# tester code
# EDIT: needs optimization

N = 100
T = 1
best_traders = np.array([])
best_profits = np.array([])

for _ in range(T):
    traders = generate_traders(N - 1)
    best_trader, best_profit = find_optimal_trader(traders)
    best_traders = np.append(best_traders, best_trader)
    best_profits = np.append(best_profits, best_profit)
    print(best_trader, '\t', best_profit)

print("average best trader:", best_traders.mean(axis = 1))
print("average best profit:", best_profits.mean())

KeyboardInterrupt: 